In [ ]:
%pip install -U trl

In [ ]:
from importlib import reload

from Trainers.trainer_classifier import ClassifierTrainer, ClassifierTrainingConfig
from Trainers.trainer_ppo import PPOTrainingConfig, PolicyPPOTrainer
from Models.model_policy import PolicyModel
from Models.model_value import ValueModel
from Models.model_reward import RewardModel
from Models.model_classifier import Classifier
import Datasets.dataset_request as dataset_request


dataset_request = reload(dataset_request)
RequestDataset = dataset_request.RequestDataset

In [ ]:
dataset = RequestDataset.load("human_requests_hh-rlhf.pt", "Qwen/Qwen3-0.6B")
dataset.truncate(0, 101)

In [ ]:
config = PPOTrainingConfig(
    output_dir="outputs/ppo_policy"
)
policy = PolicyModel("Qwen/Qwen3-0.6B")
value = ValueModel("Qwen/Qwen3-0.6B")
reward_model = RewardModel("Skywork/Skywork-Reward-V2-Qwen3-0.6B", "proxy")
judge = RewardModel("Skywork/Skywork-Reward-V2-Qwen3-4B", "judge")

In [ ]:
policy.generate_new_dataset(dataset, 4)

In [ ]:
reward_model.init_normalization(policy)
judge.init_normalization(policy)

reward_model.score_policy(policy)
judge.score_policy(policy)

In [ ]:
classfier_dataset = policy.generate_dataset_classifier(2)
train, test = classfier_dataset.split()

In [ ]:
classifier_config = ClassifierTrainingConfig(output_dir="outputs/classifiers")
classifier = Classifier("1", "Qwen/Qwen3-0.6B")

classifier_trainer = ClassifierTrainer(classifier, classifier_config)
classifier_trainer.train(train, test)